# Lab 14 Human versus AI Reasoning Audit

**Course:** Cognitive Science  
**Environment:** Google Colab  
**Estimated time:** 120-150 minutes

## Learning objectives
- Build a standardized reasoning-item table.
- Compare human and AI answers using explicit scoring.
- Classify errors without claiming identical internal mechanisms.

## Theory in brief
Behavioral comparison can reveal accuracy, consistency, and framing sensitivity. It cannot by itself show that a machine and a person use the same cognitive process.

## Ethics and data handling
Use an anonymous participant code. This classroom activity is not a diagnostic test. A student may use the simulation path instead of providing personal behavioral data.

## Pre-lab prediction
Write a directional prediction and one theoretical reason before running the experiment.

**My prediction:** คนกับ AI น่าจะได้คะแนนรวมใกล้กันในโจทย์ที่มีคำตอบชัด แต่ AI อาจพลาดโจทย์ที่ต้องระวัง base-rate หรือ conjunction เพราะมันไวต่อการเลือกคำมากกว่าเหตุผลเชิงความน่าจะเป็น


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
RNG = np.random.default_rng(42)
print('Environment ready')


Environment ready


In [2]:
items = pd.DataFrame([
    {'id':1,'type':'base_rate','question':'A description matches a stereotype, but 5 of 100 people belong to that group. Which information should dominate?','key':'base rate'},
    {'id':2,'type':'framing','question':'Program A saves 200 of 600 people; Program B has a 1/3 chance to save all and 2/3 chance to save none. Choose A or B.','key':'equivalent'},
    {'id':3,'type':'logic','question':'All artists are creative. Some teachers are artists. Does it follow that some teachers are creative?','key':'yes'},
    {'id':4,'type':'conjunction','question':'Can P(A and B) exceed P(A)?','key':'no'}])

# กรอกคำตอบ (human = ตัวเรา, ai = ทดลองถามโมเดลสองรอบ) พร้อมให้คะแนนด้วย key
items['human_answer'] = ['base rate', 'equivalent', 'yes', 'no']
items['ai_answer_run1'] = ['base rate', 'equivalent', 'yes', 'no']
items['ai_answer_run2'] = ['base rate', 'depends on values', 'yes', 'no']
def score(ans, key):
    return int(str(ans).strip().lower() == str(key).strip().lower())
items['human_score']   = [score(a,k) for a,k in zip(items.human_answer, items.key)]
items['ai_score_run1'] = [score(a,k) for a,k in zip(items.ai_answer_run1, items.key)]
items['ai_score_run2'] = [score(a,k) for a,k in zip(items.ai_answer_run2, items.key)]
items.to_csv('lab14_reasoning_audit.csv', index=False)
items


,id,type,question,key,human_answer,ai_answer_run1,ai_answer_run2,human_score,ai_score_run1,ai_score_run2
0,1,base_rate,"A description matches a stereotype, but 5 of 1...",base rate,base rate,base rate,base rate,1,1,1
1,2,framing,Program A saves 200 of 600 people; Program B h...,equivalent,equivalent,equivalent,depends on values,1,1,0
2,3,logic,All artists are creative. Some teachers are ar...,yes,yes,yes,yes,1,1,1
3,4,conjunction,Can P(A and B) exceed P(A)?,no,no,no,no,1,1,1


## Data entry protocol
Answer each item yourself before asking an AI. Use the exact same prompt twice in fresh conversations if allowed. Record model name, visible version, date, prompt, and settings. Score with the key or an instructor-approved rubric.


In [3]:
# After filling the CSV, upload it in Colab or edit the table above.
# Example analysis functions:
def summarize_audit(df):
 out={}
 for c in ['human_score','ai_score_run1','ai_score_run2']:
  if c in df and df[c].notna().any(): out[c]=df[c].mean()
 if {'ai_answer_run1','ai_answer_run2'}<=set(df): out['ai_repeat_agreement']=(df.ai_answer_run1==df.ai_answer_run2).mean()
 return out
print(summarize_audit(items))


{'human_score': np.float64(1.0), 'ai_score_run1': np.float64(1.0), 'ai_score_run2': np.float64(0.75), 'ai_repeat_agreement': np.float64(0.75)}


## Error taxonomy
Assign each wrong response to one defensible category: base-rate neglect, framing sensitivity, invalid inference, conjunction error, instruction failure, unsupported explanation, or other. Support the label with the response text.


## Interpretation and submission

**1. Main numerical result:** human score = 1.00, AI score เฉลี่ยสองรอบ = 0.875 (run1 = 1.00, run2 = 0.75), agreement ระหว่างสองรอบของ AI = 0.75

**2. Does it support the prediction:** สนับสนุนบางส่วน คนกับ AI ทำโจทย์ชัดๆ ได้เท่ากัน แต่ AI ตอบไม่นิ่งในข้อ framing (รอบสองเปลี่ยนคำตอบ) ตรงกับที่คาดว่ามันไวต่อการเลือกคำ

**3. Alternative explanation & design improvement:** ทางเลือก: การที่ AI ตอบต่างสองรอบอาจมาจากความสุ่มของโมเดล ไม่ใช่ความเข้าใจที่ต่าง / ปรับปรุง: เพิ่มจำนวนข้อและถาม AI หลายรอบด้วย prompt เดียวกันเพื่อวัดความสม่ำเสมอให้เสถียร และให้คนมากกว่า 1 คนช่วย score ตาม rubric

**4. Files:** ดาวน์โหลด notebook นี้พร้อมไฟล์ CSV ที่ถูกสร้างขึ้นทุกไฟล์เพื่อส่ง

> หมายเหตุ: error ที่พบเป็นชนิด framing sensitivity (AI รอบสองไม่ยืนยันว่า A/B สมมูลกัน)

## AI use declaration

- Tool/version: Claude (Anthropic)
- Purpose of use: ช่วยเขียนโค้ดส่วน simulation และการวิเคราะห์ และช่วยเรียบเรียงคำอธิบายผล
- Verification performed: รันโค้ดซ้ำจนไม่มี error และตรวจว่าตัวเลข/กราฟที่ได้สมเหตุสมผลตามทฤษฎีก่อนนำไปใช้
